# CS231 — So sánh 3 CNN trên cùng tập test

**Cách chạy:** chọn Runtime → Change runtime type → GPU, tải `cs231_colab_bundle.zip`
lên **My Drive/CS231/**, rồi chạy lần lượt các cell.

Notebook dùng code đã resolve conflict tại commit `8e0a5e7`, dữ liệu đã chia cố định
và checkpoint ResNet18 hiện có. Train LightWasteCNN từ đầu (tối đa 35 epoch),
MobileNetV4 pretrained (tối đa 30 epoch, 3 epoch đóng băng backbone).
Chọn checkpoint bằng validation Macro F1, tuyệt đối không chọn model bằng test.

Kết quả và checkpoint lưu vào Drive. Không dùng file MobileNetV4 demo cũ.
Nếu Colab ngắt, chạy lại notebook sẽ bỏ qua model đã hoàn tất; model đang train
sẽ train lại từ đầu (chưa hỗ trợ resume optimizer/scheduler).


In [ ]:
import sys, subprocess
import torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi tiếp tục.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
# Giữ nguyên bộ torch/torchvision do Colab cung cấp.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm>=1.0.8', 'scikit-learn>=1.3', 'scikit-image>=0.22',
                'matplotlib', 'pyyaml', 'pandas', 'joblib', 'opencv-python-headless'], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, zipfile, hashlib, shutil
BUNDLE = Path('/content/drive/MyDrive/CS231/cs231_colab_bundle.zip')
RUN = Path('/content/drive/MyDrive/CS231/comparison_20261118')
WORK = Path('/content/cs231_comparison')
assert BUNDLE.is_file(), f'Tải file ZIP vào {BUNDLE}'
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    for item in z.infolist():
        target = (WORK / item.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), 'Invalid archive path'
    z.extractall(WORK)
os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
manifest = json.loads((WORK / 'bundle_manifest.json').read_text())
for name, digest in manifest['files'].items():
    assert hashlib.sha256((WORK / name).read_bytes()).hexdigest() == digest, name
print('Verified bundle:', manifest['source_commit'])


In [ ]:
from src.member2_deeplearning.audit_data import audit_dataset
from src.base_predictor import CLASS_NAMES
DATA = WORK / 'data/standard_split'
audit = audit_dataset(DATA)
assert audit['manifest_sha256'] == manifest['dataset_manifest_sha256']
checkpoint = torch.load(WORK / 'weights/resnet18_best.pth', map_location='cpu', weights_only=True)
assert checkpoint['checkpoint_kind'] == 'trained'
assert checkpoint['model_name'] == 'resnet18'
assert checkpoint['dataset_manifest_sha256'] == audit['manifest_sha256']
assert checkpoint['class_names'] == list(CLASS_NAMES)
(RUN / 'weights').mkdir(exist_ok=True)
shutil.copy2(WORK / 'weights/resnet18_best.pth', RUN / 'weights/resnet18_best.pth')
(RUN / 'dataset_audit.json').write_text(json.dumps(audit, indent=2))
print(json.dumps(audit['counts'], indent=2))
print('ResNet18:', checkpoint['epoch'], 'epochs at selected checkpoint; val F1:', checkpoint['val_macro_f1'])
del checkpoint


## Train hai model còn thiếu
Mỗi model được chọn checkpoint riêng. Các lần chạy dùng cùng seed, dữ liệu và
augmentation; ngân sách epoch khác nhau và ResNet18 được kế thừa từ lần train cũ,
nên đây không phải thí nghiệm kiểm soát cùng ngân sách tính toán.
Không chỉnh siêu tham số sau khi đã xem kết quả test. Các cell benchmark chạy sau khi
cả hai lượt huấn luyện hoàn tất. Có thể chạy riêng từng cell train nếu thời lượng Colab hạn chế.


In [ ]:
def train_if_needed(model, epochs, freeze_epochs):
    summary = RUN / 'results/dl' / model / 'summary.json'
    weight_name = 'mobilenetv4_best.pth' if model == 'mobilenetv4_conv_small' else f'{model}_best.pth'
    weights = RUN / 'weights' / weight_name
    if summary.exists() and weights.exists():
        info = json.loads(summary.read_text())
        saved = torch.load(weights, map_location='cpu', weights_only=True)
        assert info['data_audit']['manifest_sha256'] == audit['manifest_sha256']
        assert saved['checkpoint_kind'] == 'trained' and saved['model_name'] == model
        assert saved['dataset_manifest_sha256'] == audit['manifest_sha256']
        print('Đã hoàn tất, bỏ qua:', model)
        return
    subprocess.run([sys.executable, '-u', '-m', 'src.member2_deeplearning.train_cnn',
                    '--model', model, '--epochs', str(epochs), '--freeze_epochs', str(freeze_epochs),
                    '--batch_size', '32', '--lr', '0.0003', '--patience', '7', '--workers', '2',
                    '--seed', '42', '--data', str(DATA), '--output_root', str(RUN)], check=True)

train_if_needed('light_cnn', 35, 0)


In [ ]:
train_if_needed('mobilenetv4_conv_small', 30, 3)


## Đánh giá 3 model trên cùng 716 ảnh test
Đo mặc định trên GPU Colab, batch size 1, 3 warmup, không tính Grad-CAM.
Chỉ so sánh tốc độ giữa các model trong lượt chạy này; không so trực tiếp với số CPU local.
Đổi `DEVICE = 'cpu'` nếu cần bảng CPU; cả ba model vẫn dùng cùng thiết bị.
Kết quả được ghi sau mỗi model, tránh mất toàn bộ nếu runtime ngắt.


In [ ]:
import time, platform, csv
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from src.member2_deeplearning.predictor_dl import DeepLearningPredictor
from src.member2_deeplearning.dataset_aug import EXTENSIONS
DEVICE = 'cuda'
torch.set_num_threads(2)
cv2.setNumThreads(1)
OUT = RUN / 'comparison'
OUT.mkdir(exist_ok=True)
specs = {'light_cnn': 'light_cnn_best.pth', 'resnet18': 'resnet18_best.pth',
         'mobilenetv4_conv_small': 'mobilenetv4_best.pth'}
# Validate all three checkpoints before inspecting any test results.
for name, filename in specs.items():
    ck = torch.load(RUN / 'weights' / filename, map_location='cpu', weights_only=True)
    assert ck['checkpoint_kind'] == 'trained' and ck['model_name'] == name
    assert ck['dataset_manifest_sha256'] == audit['manifest_sha256']
    assert ck['class_names'] == list(CLASS_NAMES)
    del ck
samples = [(p, label) for label, cls in enumerate(CLASS_NAMES)
           for p in sorted((DATA / 'test' / cls).iterdir()) if p.suffix.lower() in EXTENSIONS]
assert len(samples) == 716
rows = []
for name, filename in specs.items():
    model = DeepLearningPredictor(str(RUN / 'weights' / filename), device=DEVICE)
    try:
        for _ in range(3):
            model.predict(np.zeros((224, 224, 3), dtype=np.uint8), with_explanation=False)
        true, pred, latency, detail = [], [], [], []
        for path, target in samples:
            bgr = cv2.imread(str(path))
            assert bgr is not None, path
            rgb = cv2.cvtColor(cv2.resize(bgr, (224, 224), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
            result = model.predict(rgb, with_explanation=False)
            true.append(target); pred.append(result['class_id']); latency.append(result['latency_ms'])
            detail.append({'image': str(path.relative_to(DATA)), 'true': CLASS_NAMES[target],
                           'predicted': result['class_name'], 'confidence': result['confidence'],
                           'latency_ms': result['latency_ms']})
        row = {'model': name, 'n_test': len(true), 'accuracy_pct': 100*accuracy_score(true, pred),
               'macro_f1_pct': 100*f1_score(true, pred, labels=list(range(6)), average='macro', zero_division=0),
               'mean_ms': float(np.mean(latency)), 'median_ms': float(np.median(latency)),
               'p95_ms': float(np.percentile(latency, 95)), 'fps': 1000/np.mean(latency),
               'parameters': sum(p.numel() for p in model.model.parameters()),
               'checkpoint_mib': (RUN/'weights'/filename).stat().st_size/2**20, 'device': DEVICE}
        rows.append(row)
        pd.DataFrame(detail).to_csv(OUT/f'{name}_predictions.csv', index=False)
        report = classification_report(true, pred, labels=list(range(6)), target_names=CLASS_NAMES,
                                       output_dict=True, zero_division=0)
        (OUT/f'{name}_report.json').write_text(json.dumps(report, indent=2))
        fig, ax = plt.subplots(figsize=(8, 7))
        ConfusionMatrixDisplay(confusion_matrix(true, pred, labels=list(range(6))), display_labels=CLASS_NAMES).plot(
            ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
        ax.set_title(name); fig.tight_layout(); fig.savefig(OUT/f'{name}_confusion.png', dpi=160); plt.close(fig)
        pd.DataFrame(rows).to_csv(OUT/'comparison.csv', index=False)
        print(row)
    finally:
        model.close()
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
environment = {'device': DEVICE, 'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
               'python': platform.python_version(), 'dataset_manifest': audit['manifest_sha256'],
               'timing': 'batch=1; 3 warmups; normalization+forward only; excludes disk IO, resizing and Grad-CAM'}
(OUT/'environment.json').write_text(json.dumps(environment, indent=2))
display(pd.DataFrame(rows).round(3))


In [ ]:
table = pd.read_csv(OUT/'comparison.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
table.plot.bar(x='model', y=['accuracy_pct', 'macro_f1_pct'], ax=axes[0], rot=20)
axes[0].set_ylabel('%'); axes[0].set_ylim(0, 100); axes[0].set_title('Held-out test performance')
table.plot.bar(x='model', y='mean_ms', ax=axes[1], rot=20, legend=False)
axes[1].set_ylabel('ms / image'); axes[1].set_title(f'Inference latency ({DEVICE}, batch=1)')
fig.tight_layout(); fig.savefig(OUT/'comparison.png', dpi=160); plt.show()
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=(OUT/'pip-freeze.txt').open('w'), check=True)
archive = shutil.make_archive('/content/cs231_comparison_results', 'zip', OUT)
from google.colab import files
files.download(archive)
print('Checkpoint, history và báo cáo đã lưu tại:', RUN)
